# **🔍 Steel Surface Defect Detection**

## **🎯 Goal**
Classify a photo of a hot-rolled steel surface into one of 6 defect types using a CNN and transfer learning.
In manufacturing, a missed defect usually costs more than a false alarm, so the project focuses on **recall per defect class**, not just overall accuracy.

## **📦 Data**
[NEU Surface Defect Database](https://github.com/siddhartamukherjee/NEU-DET-Steel-Surface-Defect-Detection) (Song & Yan, Northeastern University), loaded from a GitHub mirror: 1,800 grayscale 200×200 images, 6 defect classes (`crazing`, `inclusion`, `patches`, `pitted_surface`, `rolled-in_scale`, `scratches`), 300 images per class. Labels come from the file names.

## **🗺️ Workflow**
1. **Load and explore the data**: build the labels from the file names, check class counts, plot sample images per class.
2. **Split first**: stratified train/validation/test split with a fixed seed, *before* any augmentation (to avoid data leakage).
3. **Build baselines**: a majority-class guess and a small CNN trained from scratch.
4. **Fine-tune a pretrained model**: ResNet18, first training only the final layer, then unfreezing the last block with a lower learning rate.
5. **Evaluate once on the test set**: confusion matrix, per-class precision, recall and F1, and macro-F1.
6. **Explain and analyze errors**: Grad-CAM heatmaps to check the model looks at the defect, then a review of the misclassified images.
7. **Test with less data**: train on 10%, 25% and 100% of the training set to see when transfer learning helps most.
8. **Deploy**: a Gradio app on Hugging Face Spaces.

### 📥 Loading the Dataset and Libraries

Before we start, we need to install the necessary Python libraries and **load the dataset**.

In [8]:
# Install the required libraries
# (torch, torchvision, pandas, numpy, matplotlib, seaborn and scikit-learn are already included in Colab)
%pip install grad-cam -q

# Install and update the watermark package to display environment and library version information
%pip install -q -U watermark

In [9]:
# ============================================================
# 📦 DEPENDENCIES
# ============================================================
import os
import random
import hashlib
import pathlib
import numpy as np
import pandas as pd
import subprocess

# ✅ Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# ✅ Deep learning
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

# ✅ Split and evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score

# ✅ Explainability (Grad-CAM)
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

In [4]:
%reload_ext watermark
%watermark -a "Maykon - Steel Defect Classification" -d -u -v -p numpy,pandas,matplotlib,seaborn,sklearn,torch,torchvision


Author: Maykon - Steel Defect Classification

Last updated: 2026-11-04

Python implementation: CPython
Python version       : 3.13.16
IPython version      : 7.34.0

numpy      : 2.4.6
pandas     : 2.2.3
matplotlib : 3.10.0
seaborn    : 0.13.2
sklearn    : 1.6.1
torch      : 2.11.0+cu130
torchvision: 0.26.0+cu130



In [10]:
# Download the dataset (GitHub mirror of the NEU Surface Defect Database)
repo_url = "https://github.com/siddhartamukherjee/NEU-DET-Steel-Surface-Defect-Detection"
path = "/content/neu"

if not os.path.exists(path):
    subprocess.run(["git", "clone", "--depth", "1", repo_url, path], check=True)

print("📦 Path to dataset files:", path)

📦 Path to dataset files: /content/neu


In [13]:
# Build a table with one row per image: the file path and its label.
# Labels come from the file name: "rolled-in_scale_12.jpg" -> "rolled-in_scale"
root = pathlib.Path(path)
rows = []
for folder in ["IMAGES", "Validation_Images"]:
    for p in sorted((root / folder).glob("*.jpg")):
        rows.append({"path": str(p), "label": p.stem.rsplit("_", 1)[0]})

df = pd.DataFrame(rows)

print(f"📷 Total images: {len(df)}")
counts = df["label"].value_counts().sort_index().rename("images").to_frame()
counts["share (%)"] = (counts["images"] / len(df) * 100).round(1)
counts

📷 Total images: 1800


,images,share (%)
label,,
crazing,300,16.7
inclusion,300,16.7
patches,300,16.7
pitted_surface,300,16.7
rolled-in_scale,300,16.7
scratches,300,16.7
